In [1]:
import pandas as pd
import numpy as np
import optuna
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
import warnings

warnings.filterwarnings('ignore')

# ==========================================
# 1. 設定・定数管理
# ==========================================
CONFIG = {
    "train_path": "data/train.csv",
    "test_path": "data/test.csv",
    "sub_path": "data/sample_submission.csv",
    "target": "Churn",
    "n_folds": 5,
    "random_state": 42,
    "n_trials": 20, 
    "weights": {"lgbm": 0.45, "xgb": 0.45, "cat": 0.10}
}

# ==========================================
# 2. 特徴量エンジニアリング関数
# ==========================================
def preprocess_data(train_df, test_df):
    X = train_df.drop(['id', CONFIG["target"]], axis=1).copy()
    y = train_df[CONFIG["target"]].map({'Yes': 1, 'No': 0})
    X_test = test_df.drop(['id'], axis=1).copy()

    services = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']

    for df in [X, X_test]:
        df['AvgMonthlyCharges'] = df['TotalCharges'] / (df['tenure'] + 1)
        df['IsNewCustomer'] = (df['tenure'] == 0).astype(int)
        
        # 修正: 特定の列(services)のみで 'Yes' をカウントする
        df['TotalServices'] = (df[services] == 'Yes').sum(axis=1)
        
        df['Contract_Payment'] = df['Contract'].astype(str) + "_" + df['PaymentMethod'].astype(str)
        df['Net_Support'] = df['InternetService'].astype(str) + "_" + df['TechSupport'].astype(str)

    cat_features = X.select_dtypes(include=['object']).columns
    for col in cat_features:
        le = LabelEncoder()
        full_labels = pd.concat([X[col], X_test[col]]).astype(str)
        le.fit(full_labels)
        X[col] = le.transform(X[col].astype(str))
        X_test[col] = le.transform(X_test[col].astype(str))
        
    return X, y, X_test

# ==========================================
# 3. 学習・最適化管理クラス
# ==========================================
class ModelTrainer:
    def __init__(self, X, y, X_test):
        self.X, self.y, self.X_test = X, y, X_test
        self.skf = StratifiedKFold(n_splits=CONFIG["n_folds"], shuffle=True, random_state=CONFIG["random_state"])

    # --- LightGBM ---
    def optimize_lgbm(self, n_trials):
        def objective(trial):
            param = {
                'objective': 'binary', 'metric': 'auc', 'verbosity': -1, 'random_state': CONFIG["random_state"],
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
                'num_leaves': trial.suggest_int('num_leaves', 20, 150),
                'feature_fraction': trial.suggest_float('feature_fraction', 0.4, 1.0),
                'bagging_fraction': trial.suggest_float('bagging_fraction', 0.4, 1.0),
                'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
                'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
            }
            X_tr, X_val, y_tr, y_val = train_test_split(self.X, self.y, test_size=0.2, stratify=self.y, random_state=42)
            
            # 修正: log_evaluationを削除し、callbacksに含める
            model = lgb.train(param, lgb.Dataset(X_tr, label=y_tr), 
                              valid_sets=[lgb.Dataset(X_val, label=y_val)],
                              num_boost_round=1000,
                              callbacks=[lgb.early_stopping(50), lgb.log_evaluation(period=0)])
            return model.best_score['valid_0']['auc']

        print("\nOptimizing LightGBM...")
        study = optuna.create_study(direction='maximize')
        study.optimize(objective, n_trials=n_trials)
        return study.best_params

    # --- XGBoost ---
    def optimize_xgb(self, n_trials):
        def objective(trial):
            param = {
                'objective': 'binary:logistic', 'eval_metric': 'auc', 'verbosity': 0, 'random_state': CONFIG["random_state"],
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
                'max_depth': trial.suggest_int('max_depth', 3, 10),
                'subsample': trial.suggest_float('subsample', 0.5, 1.0),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
                'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
            }
            X_tr, X_val, y_tr, y_val = train_test_split(self.X, self.y, test_size=0.2, stratify=self.y, random_state=42)
            dtrain, dvalid = xgb.DMatrix(X_tr, label=y_tr), xgb.DMatrix(X_val, label=y_val)
            model = xgb.train(param, dtrain, evals=[(dvalid, 'eval')], early_stopping_rounds=50, num_boost_round=1000, verbose_eval=False)
            return model.best_score

        print("\nOptimizing XGBoost...")
        study = optuna.create_study(direction='maximize')
        study.optimize(objective, n_trials=n_trials)
        return study.best_params

    # --- CatBoost ---
    def optimize_cat(self, n_trials):
        def objective(trial):
            param = {
                'objective': 'Logloss', 'eval_metric': 'AUC', 'random_seed': CONFIG["random_state"], 'logging_level': 'Silent',
                'depth': trial.suggest_int('depth', 1, 10),
                'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.01, 0.1),
                'boosting_type': trial.suggest_categorical('boosting_type', ['Ordered', 'Plain']),
                'bootstrap_type': trial.suggest_categorical('bootstrap_type', ['Bayesian', 'Bernoulli', 'MVS'])
            }
            # bootstrap_typeに依存するパラメータの調整
            if param['bootstrap_type'] == 'Bayesian':
                param['bagging_temperature'] = trial.suggest_float('bagging_temperature', 0, 10)
            elif param['bootstrap_type'] in ['Bernoulli', 'MVS']:
                param['subsample'] = trial.suggest_float('subsample', 0.1, 1.0)

            X_tr, X_val, y_tr, y_val = train_test_split(self.X, self.y, test_size=0.2, stratify=self.y, random_state=42)
            model = CatBoostClassifier(**param)
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], early_stopping_rounds=50)
            return model.get_best_score()['validation']['AUC']

        print("\nOptimizing CatBoost...")
        study = optuna.create_study(direction='maximize')
        study.optimize(objective, n_trials=n_trials)
        return study.best_params

    # --- Cross-Validation ---
    def run_cv(self, model_type, best_params):
        oof_preds = np.zeros(len(self.X))
        test_preds = np.zeros(len(self.X_test))
        
        print(f"\nRunning {CONFIG['n_folds']}-Fold CV for {model_type.upper()}...")
        for fold, (train_idx, valid_idx) in enumerate(self.skf.split(self.X, self.y)):
            X_tr, X_val = self.X.iloc[train_idx], self.X.iloc[valid_idx]
            y_tr, y_val = self.y.iloc[train_idx], self.y.iloc[valid_idx]

            if model_type == 'lgbm':
                # 修正: log_evaluationを削除し、callbacksに移動
                m = lgb.train({**best_params, 'objective': 'binary', 'metric': 'auc', 'verbosity': -1}, 
                              lgb.Dataset(X_tr, label=y_tr), 
                              valid_sets=[lgb.Dataset(X_val, label=y_val)],
                              num_boost_round=1000, 
                              callbacks=[lgb.early_stopping(50), lgb.log_evaluation(period=0)])
                oof_preds[valid_idx] = m.predict(X_val)
                test_preds += m.predict(self.X_test) / CONFIG["n_folds"]

            elif model_type == 'xgb':
                m = xgb.train({**best_params, 'objective': 'binary:logistic', 'eval_metric': 'auc'}, 
                              xgb.DMatrix(X_tr, label=y_tr), 
                              evals=[(xgb.DMatrix(X_val, label=y_val), 'eval')],
                              num_boost_round=1000, early_stopping_rounds=50, verbose_eval=False)
                oof_preds[valid_idx] = m.predict(xgb.DMatrix(X_val))
                test_preds += m.predict(xgb.DMatrix(self.X_test)) / CONFIG["n_folds"]

            elif model_type == 'cat':
                m = CatBoostClassifier(**{**best_params, 'objective': 'Logloss', 'eval_metric': 'AUC', 'logging_level': 'Silent'})
                m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], early_stopping_rounds=50)
                oof_preds[valid_idx] = m.predict_proba(X_val)[:, 1]
                test_preds += m.predict_proba(self.X_test)[:, 1] / CONFIG["n_folds"]

        print(f"-> {model_type.upper()} CV AUC: {roc_auc_score(self.y, oof_preds):.5f}")
        return oof_preds, test_preds

# ==========================================
# 4. メイン実行フロー
# ==========================================
def main():
    print("Loading data...")
    train_df = pd.read_csv(CONFIG["train_path"])
    test_df = pd.read_csv(CONFIG["test_path"])
    
    X, y, X_test = preprocess_data(train_df, test_df)
    trainer = ModelTrainer(X, y, X_test)

    # 各モデルの最適化と学習
    best_p_lgbm = trainer.optimize_lgbm(CONFIG["n_trials"])
    oof_lgbm, pred_lgbm = trainer.run_cv('lgbm', best_p_lgbm)

    best_p_xgb = trainer.optimize_xgb(CONFIG["n_trials"])
    oof_xgb, pred_xgb = trainer.run_cv('xgb', best_p_xgb)

    best_p_cat = trainer.optimize_cat(CONFIG["n_trials"])
    oof_cat, pred_cat = trainer.run_cv('cat', best_p_cat)

    # 最終ブレンディング
    w = CONFIG["weights"]
    oof_final = (oof_lgbm * w["lgbm"]) + (oof_xgb * w["xgb"]) + (oof_cat * w["cat"])
    preds_final = (pred_lgbm * w["lgbm"]) + (pred_xgb * w["xgb"]) + (pred_cat * w["cat"])
    
    print(f"\n{'='*35}")
    print(f"Final Blended CV AUC: {roc_auc_score(y, oof_final):.5f}")
    print(f"{'='*35}")

    # 提出用CSV作成
    submission = pd.read_csv(CONFIG["sub_path"])
    submission[CONFIG["target"]] = preds_final
    submission.to_csv("submission_optuna_blended_v2.csv", index=False)
    print("Success: submission_optuna_blended_v2.csv created.")

if __name__ == "__main__":
    main()

Loading data...


[I 2026-03-05 14:46:58,881] A new study created in memory with name: no-name-61b54be1-3d99-4217-8e94-1e10432caae4



Optimizing LightGBM...
Training until validation scores don't improve for 50 rounds


[I 2026-03-05 14:47:10,801] Trial 0 finished with value: 0.9168216799981577 and parameters: {'learning_rate': 0.04359302755640909, 'num_leaves': 31, 'feature_fraction': 0.5691205616321104, 'bagging_fraction': 0.644340397798137, 'bagging_freq': 6, 'min_child_samples': 52}. Best is trial 0 with value: 0.9168216799981577.


Early stopping, best iteration is:
[822]	valid_0's auc: 0.916822
Training until validation scores don't improve for 50 rounds


...


Running 5-Fold CV for XGB...


[I 2026-03-05 14:56:26,389] A new study created in memory with name: no-name-c784fa2e-e589-4794-8df3-95db780bcf2f


-> XGB CV AUC: 0.91669

Optimizing CatBoost...


[I 2026-03-05 14:56:38,277] Trial 0 finished with value: 0.9131650369316097 and parameters: {'depth': 2, 'colsample_bylevel': 0.028354455799793628, 'boosting_type': 'Plain', 'bootstrap_type': 'MVS', 'subsample': 0.10084906384586137}. Best is trial 0 with value: 0.9131650369316097.
[I 2026-03-05 14:57:25,816] Trial 1 finished with value: 0.9115614489876097 and parameters: {'depth': 1, 'colsample_bylevel': 0.06026191962865147, 'boosting_type': 'Ordered', 'bootstrap_type': 'MVS', 'subsample': 0.4091164386212299}. Best is trial 0 with value: 0.9131650369316097.
[I 2026-03-05 14:58:19,853] Trial 2 finished with value: 0.9115192414824229 and parameters: {'depth': 4, 'colsample_bylevel': 0.01763544597848955, 'boosting_type': 'Ordered', 'bootstrap_type': 'Bayesian', 'bagging_temperature': 9.31510201382785}. Best is trial 0 with value: 0.9131650369316097.
[I 2026-03-05 14:59:14,298] Trial 3 finished with value: 0.9102894299328688 and parameters: {'depth': 1, 'colsample_bylevel': 0.0131985438047


Running 5-Fold CV for CAT...
-> CAT CV AUC: 0.91511

Final Blended CV AUC: 0.91667
Success: submission_optuna_blended_v2.csv created.
